In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.6 MB/s eta 0:00:00


In [4]:
from torch_geometric.datasets import UPFD
import collections

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"

for name in ["politifact", "gossipcop"]:
    for split in ["train", "val", "test"]:
        ds = UPFD(root=ROOT, name=name, feature="bert", split=split)
        labels = [int(g.y) for g in ds]
        print(name, split, "| graphs:", len(ds),
              "| labels:", dict(collections.Counter(labels)),
              "| feat dim:", ds.num_features)

politifact train | graphs: 62 | labels: {0: 36, 1: 26} | feat dim: 768
politifact val | graphs: 31 | labels: {1: 18, 0: 13} | feat dim: 768
politifact test | graphs: 221 | labels: {1: 113, 0: 108} | feat dim: 768


Extracting /content/drive/MyDrive/fake-news-sna/data/UPFD/gossipcop/raw/data.zip
Processing...
Done!


gossipcop train | graphs: 1092 | labels: {0: 557, 1: 535} | feat dim: 768
gossipcop val | graphs: 546 | labels: {1: 281, 0: 265} | feat dim: 768
gossipcop test | graphs: 3826 | labels: {1: 1916, 0: 1910} | feat dim: 768


In [5]:
import torch
from torch_geometric.datasets import UPFD

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"
ds = UPFD(root=ROOT, name="politifact", feature="bert", split="train")

for i in range(3):
    g = ds[i]
    n = g.num_nodes
    out_deg = torch.bincount(g.edge_index[0], minlength=n)
    in_deg = torch.bincount(g.edge_index[1], minlength=n)
    print(f"graph {i}: nodes={n}, edges={g.num_edges}, label={int(g.y)}")
    print("  node 0 -> out-degree:", int(out_deg[0]), "| in-degree:", int(in_deg[0]))
    print("  max out-degree node:", int(out_deg.argmax()),
          "| max in-degree node:", int(in_deg.argmax()))

graph 0: nodes=72, edges=71, label=0
  node 0 -> out-degree: 27 | in-degree: 0
  max out-degree node: 0 | max in-degree node: 1
graph 1: nodes=32, edges=31, label=1
  node 0 -> out-degree: 9 | in-degree: 0
  max out-degree node: 0 | max in-degree node: 1
graph 2: nodes=312, edges=311, label=0
  node 0 -> out-degree: 82 | in-degree: 0
  max out-degree node: 0 | max in-degree node: 1


In [6]:
import numpy as np
from torch_geometric.datasets import UPFD
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"

def root_features(ds):
    X = np.stack([g.x[0].numpy() for g in ds])   # node 0 = news node (verified)
    y = np.array([int(g.y) for g in ds])
    return X, y

rows = []
for name in ["politifact", "gossipcop"]:
    tr = UPFD(root=ROOT, name=name, feature="bert", split="train")
    te = UPFD(root=ROOT, name=name, feature="bert", split="test")
    Xtr, ytr = root_features(tr)
    Xte, yte = root_features(te)

    models = {
        "LogReg": LogisticRegression(max_iter=2000),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    }
    for mname, m in models.items():
        m.fit(Xtr, ytr)
        pred = m.predict(Xte)
        acc = accuracy_score(yte, pred)
        f1 = f1_score(yte, pred)
        rows.append((name, mname, round(acc, 4), round(f1, 4)))
        print(f"{name:11s} {mname:13s} acc={acc:.4f} f1={f1:.4f}")

import pandas as pd
pd.DataFrame(rows, columns=["dataset", "model", "accuracy", "f1"]) \
  .to_csv("/content/drive/MyDrive/fake-news-sna/results/text_only_baseline.csv", index=False)

politifact  LogReg        acc=0.8009 f1=0.7905
politifact  RandomForest  acc=0.7873 f1=0.7565
gossipcop   LogReg        acc=0.7365 f1=0.7228
gossipcop   RandomForest  acc=0.7256 f1=0.6942


In [7]:
import pandas as pd
import numpy as np
from torch_geometric.datasets import UPFD

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"
rows = []
for name in ["politifact", "gossipcop"]:
    graphs = []
    for split in ["train", "val", "test"]:
        graphs += list(UPFD(root=ROOT, name=name, feature="bert", split=split))
    nodes = np.array([g.num_nodes for g in graphs])
    ys = np.array([int(g.y) for g in graphs])
    rows.append({
        "dataset": name,
        "graphs": len(graphs),
        "real(0)": int((ys == 0).sum()),
        "fake(1)": int((ys == 1).sum()),
        "avg_nodes": round(nodes.mean(), 1),
        "median_nodes": int(np.median(nodes)),
        "max_nodes": int(nodes.max()),
        "min_nodes": int(nodes.min()),
    })

stats = pd.DataFrame(rows)
print(stats.to_string(index=False))
stats.to_csv("/content/drive/MyDrive/fake-news-sna/results/dataset_stats.csv", index=False)

   dataset  graphs  real(0)  fake(1)  avg_nodes  median_nodes  max_nodes  min_nodes
politifact     314      157      157      130.7            78        497          3
 gossipcop    5464     2732     2732       57.5            49        199          3


In [9]:
import numpy as np, pandas as pd, networkx as nx
from networkx.algorithms.community import louvain_communities, modularity
from torch_geometric.datasets import UPFD

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"
OUT  = "/content/drive/MyDrive/fake-news-sna/results"

def net_features(g):
    n = g.num_nodes
    ei = g.edge_index.numpy()
    D = nx.DiGraph(); D.add_nodes_from(range(n)); D.add_edges_from(zip(ei[0], ei[1]))
    U = D.to_undirected()

    depth = nx.single_source_shortest_path_length(D, 0)
    d = np.array(list(depth.values()))
    levels = np.bincount(d)
    out_deg = np.array([D.out_degree(i) for i in range(n)])

    deg_c = np.array(list(nx.degree_centrality(U).values()))
    btw   = np.array(list(nx.betweenness_centrality(U).values()))
    clo   = np.array(list(nx.closeness_centrality(U).values()))
    pr    = np.array(list(nx.pagerank(U).values()))

    comms = louvain_communities(U, seed=42)
    mod = modularity(U, comms)

    return {
        "n_nodes": n, "depth_max": d.max(), "depth_mean": d.mean(),
        "width_max": levels.max(), "root_out_deg": out_deg[0],
        "max_out_deg": out_deg.max(), "leaf_frac": (out_deg == 0).mean(),
        "deg_c_max": deg_c.max(), "deg_c_std": deg_c.std(),
        "btw_max": btw.max(), "btw_mean": btw.mean(),
        "clo_mean": clo.mean(), "pr_max": pr.max(), "pr_std": pr.std(),
        "n_comm": len(comms), "modularity": mod,
    }

all_rows = []
for name in ["politifact", "gossipcop"]:
    for split in ["train", "val", "test"]:
        ds = UPFD(root=ROOT, name=name, feature="bert", split=split)
        for g in ds:
            r = net_features(g)
            r.update({"dataset": name, "split": split, "label": int(g.y)})
            all_rows.append(r)
        print("done", name, split)

feats = pd.DataFrame(all_rows)
feats.to_csv(f"{OUT}/network_features.csv", index=False)
print(feats.groupby(["dataset", "label"])[["n_nodes", "depth_max", "width_max", "modularity"]].mean().round(3))

done politifact train
done politifact val
done politifact test
done gossipcop train
done gossipcop val
done gossipcop test
                  n_nodes  depth_max  width_max  modularity
dataset    label                                           
gossipcop  0       76.164      2.481     62.157       0.207
           1       38.866      2.543     27.899       0.298
politifact 0      114.917      3.624     71.764       0.417
           1      146.573      3.885     87.656       0.472


In [10]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

feats = pd.read_csv(f"{OUT}/network_features.csv")
cols = [c for c in feats.columns if c not in ("dataset", "split", "label")]

rows = []
for name in ["politifact", "gossipcop"]:
    tr = feats[(feats.dataset == name) & (feats.split == "train")]
    te = feats[(feats.dataset == name) & (feats.split == "test")]
    models = {
        "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    }
    for mname, m in models.items():
        m.fit(tr[cols], tr.label)
        p = m.predict(te[cols])
        acc, f1 = accuracy_score(te.label, p), f1_score(te.label, p)
        rows.append((name, mname, round(acc, 4), round(f1, 4)))
        print(f"{name:11s} {mname:13s} acc={acc:.4f} f1={f1:.4f}")

pd.DataFrame(rows, columns=["dataset", "model", "accuracy", "f1"]).to_csv(f"{OUT}/network_only_baseline.csv", index=False)

politifact  LogReg        acc=0.5520 f1=0.4277
politifact  RandomForest  acc=0.5294 f1=0.4157
gossipcop   LogReg        acc=0.8220 f1=0.8214
gossipcop   RandomForest  acc=0.8053 f1=0.8073


In [11]:
import numpy as np, pandas as pd
from torch_geometric.datasets import UPFD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"
OUT  = "/content/drive/MyDrive/fake-news-sna/results"
feats = pd.read_csv(f"{OUT}/network_features.csv")
net_cols = [c for c in feats.columns if c not in ("dataset", "split", "label")]

def get_split(name, split, cols_mode):
    ds = UPFD(root=ROOT, name=name, feature="bert", split=split)
    nf = feats[(feats.dataset == name) & (feats.split == split)].reset_index(drop=True)
    assert len(nf) == len(ds)
    y = np.array([int(g.y) for g in ds])
    assert (nf.label.values == y).all()          # confirms row alignment
    if cols_mode == "size":
        X = nf[["n_nodes"]].values
    elif cols_mode == "combined":
        T = np.stack([g.x[0].numpy() for g in ds])
        X = np.hstack([T, nf[net_cols].values])
    return X, y

rows = []
for name in ["politifact", "gossipcop"]:
    for mode in ["size", "combined"]:
        Xtr, ytr = get_split(name, "train", mode)
        Xte, yte = get_split(name, "test", mode)
        models = {
            "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
            "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
        }
        for mname, m in models.items():
            m.fit(Xtr, ytr)
            p = m.predict(Xte)
            acc, f1 = accuracy_score(yte, p), f1_score(yte, p)
            rows.append((name, mode, mname, round(acc, 4), round(f1, 4)))
            print(f"{name:11s} {mode:9s} {mname:13s} acc={acc:.4f} f1={f1:.4f}")

pd.DataFrame(rows, columns=["dataset", "features", "model", "accuracy", "f1"]) \
  .to_csv(f"{OUT}/size_and_combined.csv", index=False)

politifact  size      LogReg        acc=0.4887 f1=0.0000
politifact  size      RandomForest  acc=0.5068 f1=0.4044
politifact  combined  LogReg        acc=0.8281 f1=0.8208
politifact  combined  RandomForest  acc=0.7647 f1=0.7347
gossipcop   size      LogReg        acc=0.7297 f1=0.7398
gossipcop   size      RandomForest  acc=0.7044 f1=0.6944
gossipcop   combined  LogReg        acc=0.8087 f1=0.8098
gossipcop   combined  RandomForest  acc=0.8398 f1=0.8349


In [12]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

OUT = "/content/drive/MyDrive/fake-news-sna/results"
feats = pd.read_csv(f"{OUT}/network_features.csv")

groups = {
    "structure":  ["n_nodes","depth_max","depth_mean","width_max","root_out_deg","max_out_deg","leaf_frac"],
    "centrality": ["deg_c_max","deg_c_std","btw_max","btw_mean","clo_mean","pr_max","pr_std"],
    "community":  ["n_comm","modularity"],
}
allc = sum(groups.values(), [])
configs = {"all": allc}
for g in groups:
    configs[f"all - {g}"] = [c for c in allc if c not in groups[g]]
for g in groups:
    configs[f"only {g}"] = groups[g]

rows = []
for name in ["politifact", "gossipcop"]:
    tr = feats[(feats.dataset == name) & (feats.split == "train")]
    te = feats[(feats.dataset == name) & (feats.split == "test")]
    for cname, cols in configs.items():
        accs = []
        for seed in range(5):
            m = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)
            m.fit(tr[cols], tr.label)
            accs.append(accuracy_score(te.label, m.predict(te[cols])))
        rows.append((name, cname, round(np.mean(accs), 4), round(np.std(accs), 4)))
        print(f"{name:11s} {cname:22s} acc={np.mean(accs):.4f} ± {np.std(accs):.4f}")

pd.DataFrame(rows, columns=["dataset", "features", "acc_mean", "acc_std"]) \
  .to_csv(f"{OUT}/ablation_network_groups.csv", index=False)

politifact  all                    acc=0.5131 ± 0.0074
politifact  all - structure        acc=0.5186 ± 0.0061
politifact  all - centrality       acc=0.5167 ± 0.0097
politifact  all - community        acc=0.5131 ± 0.0068
politifact  only structure         acc=0.5176 ± 0.0097
politifact  only centrality        acc=0.4887 ± 0.0149
politifact  only community         acc=0.5276 ± 0.0106
gossipcop   all                    acc=0.8075 ± 0.0010
gossipcop   all - structure        acc=0.8048 ± 0.0010
gossipcop   all - centrality       acc=0.8108 ± 0.0006
gossipcop   all - community        acc=0.8013 ± 0.0012
gossipcop   only structure         acc=0.8028 ± 0.0011
gossipcop   only centrality        acc=0.7859 ± 0.0013
gossipcop   only community         acc=0.6790 ± 0.0014


In [13]:
import torch, numpy as np, pandas as pd
import torch.nn.functional as F
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool
from torch_geometric.utils import to_undirected
from sklearn.metrics import accuracy_score, f1_score

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"
OUT  = "/content/drive/MyDrive/fake-news-sna/results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

class Net(torch.nn.Module):
    def __init__(self, in_dim, hid=128):
        super().__init__()
        self.c1 = SAGEConv(in_dim, hid)
        self.c2 = SAGEConv(hid, hid)
        self.lin = torch.nn.Linear(hid * 2, 2)
    def forward(self, x, edge_index, batch):
        ei = to_undirected(edge_index)
        x = F.relu(self.c1(x, ei))
        x = F.dropout(x, 0.5, self.training)
        x = F.relu(self.c2(x, ei))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        x = F.dropout(x, 0.5, self.training)
        return self.lin(x)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for d in loader:
        d = d.to(device)
        P.append(model(d.x, d.edge_index, d.batch).argmax(1).cpu()); Y.append(d.y.cpu())
    p, y = torch.cat(P).numpy(), torch.cat(Y).numpy()
    return accuracy_score(y, p), f1_score(y, p)

rows = []
for name in ["politifact", "gossipcop"]:
    tr = UPFD(root=ROOT, name=name, feature="bert", split="train")
    va = UPFD(root=ROOT, name=name, feature="bert", split="val")
    te = UPFD(root=ROOT, name=name, feature="bert", split="test")
    bs = 32 if name == "politifact" else 128
    trl = DataLoader(tr, batch_size=bs, shuffle=True)
    val = DataLoader(va, batch_size=256)
    tel = DataLoader(te, batch_size=256)

    for seed in range(3):
        torch.manual_seed(seed); np.random.seed(seed)
        model = Net(tr.num_features).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)
        best_val, best_test = -1, None
        for epoch in range(60):
            model.train()
            for d in trl:
                d = d.to(device); opt.zero_grad()
                F.cross_entropy(model(d.x, d.edge_index, d.batch), d.y).backward()
                opt.step()
            v_acc, _ = evaluate(model, val)
            if v_acc > best_val:
                best_val = v_acc
                best_test = evaluate(model, tel)
        rows.append((name, seed, round(best_val, 4), round(best_test[0], 4), round(best_test[1], 4)))
        print(f"{name:11s} seed={seed} val_acc={best_val:.4f} test_acc={best_test[0]:.4f} test_f1={best_test[1]:.4f}")

df = pd.DataFrame(rows, columns=["dataset", "seed", "val_acc", "test_acc", "test_f1"])
df.to_csv(f"{OUT}/gnn_graphsage.csv", index=False)
print(df.groupby("dataset")[["test_acc", "test_f1"]].agg(["mean", "std"]).round(4))

device: cuda


/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

politifact  seed=0 val_acc=0.8710 test_acc=0.8235 test_f1=0.8152


/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

politifact  seed=1 val_acc=0.8387 test_acc=0.8054 test_f1=0.8139


...

gossipcop   seed=0 val_acc=0.9670 test_acc=0.9553 test_f1=0.9555


/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

gossipcop   seed=1 val_acc=0.9652 test_acc=0.9556 test_f1=0.9558


/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

gossipcop   seed=2 val_acc=0.9597 test_acc=0.9462 test_f1=0.9475
           test_acc         test_f1        
               mean     std    mean     std
dataset                                    
gossipcop    0.9524  0.0053  0.9529  0.0047
politifact   0.8220  0.0159  0.8192  0.0081


In [14]:
import warnings; warnings.filterwarnings("ignore")
import torch, numpy as np, pandas as pd
import torch.nn.functional as F
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool
from torch_geometric.utils import to_undirected
from sklearn.metrics import accuracy_score, f1_score

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"
OUT  = "/content/drive/MyDrive/fake-news-sna/results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Net(torch.nn.Module):
    def __init__(self, in_dim, hid=128):
        super().__init__()
        self.c1 = SAGEConv(in_dim, hid); self.c2 = SAGEConv(hid, hid)
        self.lin = torch.nn.Linear(hid * 2, 2)
    def forward(self, x, edge_index, batch):
        ei = to_undirected(edge_index)
        x = F.relu(self.c1(x, ei)); x = F.dropout(x, 0.5, self.training)
        x = F.relu(self.c2(x, ei))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.lin(F.dropout(x, 0.5, self.training))

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for d in loader:
        d = d.to(device)
        P.append(model(d.x, d.edge_index, d.batch).argmax(1).cpu()); Y.append(d.y.cpu())
    p, y = torch.cat(P).numpy(), torch.cat(Y).numpy()
    return accuracy_score(y, p), f1_score(y, p)

def masked(ds, mode):
    out = []
    for g in ds:
        g = g.clone()
        if mode == "root_only":
            g.x[1:] = 0          # keep news node (index 0) only
        elif mode == "users_only":
            g.x[0] = 0           # drop news node features
        out.append(g)
    return out

rows = []
for name in ["politifact", "gossipcop"]:
    raw = {s: UPFD(root=ROOT, name=name, feature="bert", split=s) for s in ["train", "val", "test"]}
    bs = 32 if name == "politifact" else 128
    for mode in ["root_only", "users_only"]:
        tr, va, te = (masked(raw[s], mode) for s in ["train", "val", "test"])
        trl = DataLoader(tr, batch_size=bs, shuffle=True)
        val = DataLoader(va, batch_size=256); tel = DataLoader(te, batch_size=256)
        for seed in range(3):
            torch.manual_seed(seed); np.random.seed(seed)
            model = Net(768).to(device)
            opt = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)
            best_val, best_test = -1, None
            for epoch in range(60):
                model.train()
                for d in trl:
                    d = d.to(device); opt.zero_grad()
                    F.cross_entropy(model(d.x, d.edge_index, d.batch), d.y).backward()
                    opt.step()
                v_acc, _ = evaluate(model, val)
                if v_acc > best_val:
                    best_val, best_test = v_acc, evaluate(model, tel)
            rows.append((name, mode, seed, round(best_val, 4), round(best_test[0], 4), round(best_test[1], 4)))
            print(f"{name:11s} {mode:10s} seed={seed} val={best_val:.4f} test_acc={best_test[0]:.4f}")

df = pd.DataFrame(rows, columns=["dataset", "mode", "seed", "val_acc", "test_acc", "test_f1"])
df.to_csv(f"{OUT}/gnn_input_ablation.csv", index=False)
print(df.groupby(["dataset", "mode"])[["test_acc", "test_f1"]].agg(["mean", "std"]).round(4))

politifact  root_only  seed=0 val=0.9032 test_acc=0.8235
politifact  root_only  seed=1 val=0.8710 test_acc=0.8371
politifact  root_only  seed=2 val=0.8710 test_acc=0.8235
politifact  users_only seed=0 val=0.8065 test_acc=0.7692
politifact  users_only seed=1 val=0.7742 test_acc=0.7557
politifact  users_only seed=2 val=0.7742 test_acc=0.7557
gossipcop   root_only  seed=0 val=0.7747 test_acc=0.7891
gossipcop   root_only  seed=1 val=0.8626 test_acc=0.8502
gossipcop   root_only  seed=2 val=0.8315 test_acc=0.8259
gossipcop   users_only seed=0 val=0.9744 test_acc=0.9712
gossipcop   users_only seed=1 val=0.9762 test_acc=0.9710
gossipcop   users_only seed=2 val=0.9744 test_acc=0.9689
                      test_acc         test_f1        
                          mean     std    mean     std
dataset    mode                                       
gossipcop  root_only    0.8217  0.0308  0.8362  0.0165
           users_only   0.9704  0.0013  0.9707  0.0011
politifact root_only    0.8280  0.0079  0

In [15]:
import warnings; warnings.filterwarnings("ignore")
import torch, numpy as np, pandas as pd, networkx as nx
import torch.nn.functional as F
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool
from torch_geometric.utils import to_undirected
from sklearn.metrics import accuracy_score, f1_score

ROOT = "/content/drive/MyDrive/fake-news-sna/data/UPFD"
OUT  = "/content/drive/MyDrive/fake-news-sna/results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Net(torch.nn.Module):
    def __init__(self, in_dim, hid=64):
        super().__init__()
        self.c1 = SAGEConv(in_dim, hid); self.c2 = SAGEConv(hid, hid)
        self.lin = torch.nn.Linear(hid * 2, 2)
    def forward(self, x, edge_index, batch):
        ei = to_undirected(edge_index)
        x = F.relu(self.c1(x, ei)); x = F.dropout(x, 0.3, self.training)
        x = F.relu(self.c2(x, ei))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.lin(F.dropout(x, 0.3, self.training))

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for d in loader:
        d = d.to(device)
        P.append(model(d.x, d.edge_index, d.batch).argmax(1).cpu()); Y.append(d.y.cpu())
    p, y = torch.cat(P).numpy(), torch.cat(Y).numpy()
    return accuracy_score(y, p), f1_score(y, p)

def structural(ds):
    out = []
    for g in ds:
        g = g.clone(); n = g.num_nodes
        ei = g.edge_index.numpy()
        D = nx.DiGraph(); D.add_nodes_from(range(n)); D.add_edges_from(zip(ei[0], ei[1]))
        depth = nx.single_source_shortest_path_length(D, 0)
        is_root = torch.zeros(n); is_root[0] = 1
        outdeg = torch.tensor([D.out_degree(i) for i in range(n)], dtype=torch.float)
        dep = torch.tensor([depth.get(i, 0) for i in range(n)], dtype=torch.float)
        g.x = torch.stack([is_root, torch.log1p(outdeg), dep], dim=1)
        out.append(g)
    return out

rows = []
for name in ["politifact", "gossipcop"]:
    tr, va, te = (structural(UPFD(root=ROOT, name=name, feature="bert", split=s))
                  for s in ["train", "val", "test"])
    bs = 32 if name == "politifact" else 128
    trl = DataLoader(tr, batch_size=bs, shuffle=True)
    val = DataLoader(va, batch_size=256); tel = DataLoader(te, batch_size=256)
    for seed in range(3):
        torch.manual_seed(seed); np.random.seed(seed)
        model = Net(3).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)
        best_val, best_test = -1, None
        for epoch in range(60):
            model.train()
            for d in trl:
                d = d.to(device); opt.zero_grad()
                F.cross_entropy(model(d.x, d.edge_index, d.batch), d.y).backward()
                opt.step()
            v_acc, _ = evaluate(model, val)
            if v_acc > best_val:
                best_val, best_test = v_acc, evaluate(model, tel)
        rows.append((name, "structure_only", seed, round(best_val, 4), round(best_test[0], 4), round(best_test[1], 4)))
        print(f"{name:11s} structure_only seed={seed} val={best_val:.4f} test_acc={best_test[0]:.4f}")

df = pd.DataFrame(rows, columns=["dataset", "mode", "seed", "val_acc", "test_acc", "test_f1"])
df.to_csv(f"{OUT}/gnn_structure_only.csv", index=False)
print(df.groupby(["dataset", "mode"])[["test_acc", "test_f1"]].agg(["mean", "std"]).round(4))

politifact  structure_only seed=0 val=0.6452 test_acc=0.5430
politifact  structure_only seed=1 val=0.5484 test_acc=0.5113
politifact  structure_only seed=2 val=0.6452 test_acc=0.5520
gossipcop   structure_only seed=0 val=0.8370 test_acc=0.8108
gossipcop   structure_only seed=1 val=0.8333 test_acc=0.8029
gossipcop   structure_only seed=2 val=0.8352 test_acc=0.7959
                          test_acc         test_f1        
                              mean     std    mean     std
dataset    mode                                           
gossipcop  structure_only   0.8032  0.0075  0.7852  0.0108
politifact structure_only   0.5354  0.0214  0.4963  0.2833
